In [1]:
from deepdespeckling.utils.load_cosar import cos2mat
from deepdespeckling.utils.constants import PATCH_SIZE, STRIDE_SIZE
from deepdespeckling.merlin.merlin_denoiser import MerlinDenoiser
from PIL import Image
import numpy as np
import os
import glob
from deepdespeckling.utils.utils import (crop_image, get_cropping_coordinates, load_sar_image, preprocess_and_store_sar_images_from_coordinates,
                                         create_empty_folder_in_directory, preprocess_and_store_sar_images)


In [2]:
PATCH_SIZE, STRIDE_SIZE

(256, 254)

# 1. Inference with MERLIN

# 1.1. Despeckle one image with MERLIN

In [3]:
from deepdespeckling.utils.load_cosar import cos2mat
from deepdespeckling.utils.constants import PATCH_SIZE, STRIDE_SIZE
from deepdespeckling.merlin.merlin_denoiser import MerlinDenoiser

# Path to one image (cos or npy file)
# The image shape should be (image_height, image_width, 2)
image_path = "../data/data_merlin/noisy_merlin.npy"
# Model name, can be "spotlight", "stripmap" or "Sentinel-TOPS"
model_name = "spotlight"
symetrise = True
image = cos2mat(image_path).astype(np.float32)

# As an alternative, you can load your image by being sure to put in the following format:
# The shape of image is (image_height, image_width, 2), where image[:,:,0] contains the real part and image[:,:,1] contains the imaginary part of the SLC SAR image.
denoiser = MerlinDenoiser(model_name=model_name, symetrise=symetrise)
denoised_image = denoiser.denoise_image(image, patch_size=PATCH_SIZE, stride_size=STRIDE_SIZE)

Converting CoSAR to numpy array of size [ncolumns,nlines,2]
Reading image in CoSAR format.  ncolumns=164567435  nlines=1684370275


MemoryError: Unable to allocate 1.92 EiB for an array with shape (1684370275, 164567435) and data type complex64

In [4]:
import numpy as np
from deepdespeckling.utils.constants import PATCH_SIZE, STRIDE_SIZE
from deepdespeckling.merlin.merlin_denoiser import MerlinDenoiser

# Path to one image (cos or npy file)
# The image shape should be (image_height, image_width, 2) assuming the real/imaginary part are in the 2 channels.
image_path = "../data/data_merlin/noisy_merlin.npy"
image = np.load(image_path).astype(np.float32)  # shape: (128, 128, 2)

# Model name can be "spotlight", "stripmap", or "Sentinel-TOPS"
model_name = "spotlight"
symetrise = True

# Run denoising
denoiser = MerlinDenoiser(model_name=model_name, symetrise=symetrise)
denoised_image = denoiser.denoise_image(image, patch_size=PATCH_SIZE, stride_size=STRIDE_SIZE)

print("Denoised image keys:", denoised_image.keys())


100%|██████████| 1/1 [00:00<00:00,  3.89it/s]

Denoised image keys: dict_keys(['noisy', 'denoised'])


# 1.2. Despeckle a set of images using MERLIN 

### 1.2.1 Despeckle fullsize images

In [5]:
from deepdespeckling.despeckling import despeckle

# Path to a folder of several images (cos or npy files)
# The folder contains images of shape (image_height, image_width, 2)
images_folder_path = "../data/data_sar/" 
print(image_path)

# Folder where results are stored
destination_directory = "../results/results_data_sar/"
if not os.path.exists(destination_directory): 
    os.makedirs(destination_directory, exist_ok=True)

despeckle(images_folder_path, destination_directory, model_name="spotlight", symetrise=True)

../data/data_merlin/noisy_merlin.npy


100%|██████████| 7/7 [00:07<00:00,  1.13s/it]


### 1.2.2 Despeckle parts of images using custom coordinates

In [6]:
from deepdespeckling.despeckling import despeckle_from_coordinates

# Path to a folder of several images image (cos or npy files)
# The folder contains images of shape (image_height, image_width, 2)
images_folder_path = "../data/data_sar/"
# Folder where results are stored
destination_directory = "../results/results_data_sar"
# Coordinates of the subparts of the images to be despeckled
coordinates_dict = {'x_start':0,'y_start':0,'x_end':512,'y_end':512}

despeckle_from_coordinates(images_folder_path, coordinates_dict, destination_directory, model_name="spotlight", symetrise=True)

100%|██████████| 7/7 [00:08<00:00,  1.16s/it]


### 1.2.3 Despeckle parts of images using a crop tool

In [ ]:
from deepdespeckling.despeckling import despeckle_from_crop

# Path to a folder of several images image (cos or npy files)
# The folder contains images of shape (image_height, image_width, 2)
images_folder_path = "../data/data_sar/" 
# Folder where results are stored
destination_directory = "../results"
fixed = True # it will crop a 256*256 image from the position of your click
# fixed = False you will draw free-handly the area of your interest

despeckle_from_crop(images_folder_path, destination_directory, model_name="spotlight", fixed=False, symetrise=True)

# 2.Inference with SAR2SAR for Sentinel-1 Ground Range Detected (GRD) images

## 2.1. Despeckle one image with SAR2SAR

In [1]:
from deepdespeckling.utils.load_cosar import cos2mat
from deepdespeckling.utils.constants import PATCH_SIZE, STRIDE_SIZE
from deepdespeckling.sar2sar.sar2sar_denoiser import Sar2SarDenoiser
import numpy as np

# Load you image
# NB: image must be in AMPLITUDE format and have the following shape: (image_height, image_width)
image_path = "../data/data_amplitude/noisy_amplitude.npy"
image = np.load(image_path)  # ✅ Load image from file
print(image.shape)

# Denoise the image with SAR2SAR
denoiser = Sar2SarDenoiser()
denoised_image = denoiser.denoise_image(
                image, patch_size=PATCH_SIZE, stride_size=STRIDE_SIZE)
denoised_image.keys()

(605, 1600)
/app/deepdespeckling/sar2sar/saved_model/sar2sar.pth


100%|██████████| 3/3 [00:01<00:00,  1.73it/s]


dict_keys(['noisy', 'denoised'])

## 2.2. Despeckle a set of images using SAR2SAR

In [2]:
from deepdespeckling.despeckling import despeckle
import os 

# Path to a folder of several images (tiff or npy files)
image_paths_amplitude = "../data/data_amplitude/"

# NEED TO CHECK THE DATA has input shape (image_height, image_width)
# Folder where results are stored
destination_directory = "../results/results_amplitude"
if not os.path.exists(destination_directory):
    os.makedirs(destination_directory, exist_ok = True)

despeckle(image_paths_amplitude, destination_directory, model_name="sar2sar")

/app/deepdespeckling/sar2sar/saved_model/sar2sar.pth


ValueError: Unsupported image shape: (256, 256, 1)